#  Assignment 2 – Conversational AI System  
This notebook is my working prototype for **Assignment 2** of the Deploying AI course.

In this part, I’m starting with a **basic connection to OpenAI**.  
Later, I’ll add multiple services (API calls, semantic query, and more) inside a **chat interface**.

Let’s begin step-by-step — just loading the API key and sending a simple test message.


In [27]:
# --- Finance Assistant Chat App (Gradio version) ---
import os
import gradio as gr
from openai import OpenAI

# Initialize OpenAI client (uses your .secrets key)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Import the three services
from tools_api import get_stock_price       # Service 1
from tools_semantic import search_docs      # Service 2
from tools_custom import explain_simple     # Service 3
from prompts import system_prompt           # optional personality


In [64]:
from memory import add_to_memory, get_recent_memory, clear_memory
import gradio as gr


In [45]:
# --- import the simulated MCP service ---
from tools_mcp_finance import yahoo_finance_search
import asyncio


In [28]:
restricted_topics = ["cat", "cats", "dog", "dogs", "horoscope", "zodiac", "taylor swift"]

def contains_restricted_topic(text):
    """Return True if user mentions a forbidden topic."""
    return any(topic in text.lower() for topic in restricted_topics)


In [29]:
def chat_with_bot(user_input, history):
    """Main chat routing function for Gradio interface."""
    
    if contains_restricted_topic(user_input):
        response = "🚫 Sorry, I can’t discuss that topic. Let’s focus on finance instead."
    elif "price" in user_input or "stock" in user_input:
        response = get_stock_price(user_input)      # API service
    elif "meaning" in user_input or "define" in user_input:
        response = search_docs(user_input)           # Semantic service
    else:
        response = explain_simple(user_input)        # Custom service
    
    # Append both turns to chat history
    history.append((user_input, response))
    return history, history


In [5]:
# %% Step 1: Load API key from .secrets file
from dotenv import load_dotenv
import os

In [6]:

# Load environment variables from the .secrets file in the parent folder
load_dotenv("../.secrets")

# Retrieve the OpenAI API key
api_key = os.getenv("OPENAI_API_KEY")

In [7]:
# Simple check
if api_key:
    print(" API key loaded successfully!")
else:
    print("Could not find OPENAI_API_KEY. Please check your .secrets file.")

 API key loaded successfully!


## Step 2 – Connect to OpenAI
Here we create a simple client connection using the official `openai` package.

If the connection works, we’ll get a confirmation message.

In [8]:
# %% Step 2: Connect to OpenAI client
from openai import OpenAI

client = OpenAI(api_key=api_key)
print("OpenAI client connected!")


OpenAI client connected!


## Step 3 – Send a simple test message
To verify that the connection works, let's send a short prompt.  
This will also show how message formatting works (`system`, `user`, and `assistant` roles).


In [9]:
# %% Step 3: Test a simple chat completion
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a friendly study assistant."},
        {"role": "user", "content": "Say hello and tell me one fun fact about AI."}
    ]
)

# Extract the assistant’s reply
bot_message = response.choices[0].message.content
print("🤖 Assistant says:", bot_message)


🤖 Assistant says: Hello! One fun fact about AI is that the first AI program, called the Logic Theorist, was created in 1955 by Allen Newell and Herbert A. Simon. It was designed to mimic the problem-solving skills of a human mathematician and was able to prove theorems from *Principia Mathematica*, a landmark work in mathematical logic. This marked the beginning of AI as a field of study!


**What we achieved so far**

- The system can load secrets safely from `.secrets`.  
- We verified that OpenAI connection works.  
- The assistant responded to a basic test message.

Next step will be to **add memory** so the assistant can keep track of the conversation.


## 🗣️ Step 4 – Add Simple Chat Memory

To make our assistant conversational, we’ll:
1. Keep all previous messages in a list called `conversation_history`.
2. Send the entire conversation to the model every time we ask something new.
3. Print the assistant’s reply and update the list.

This is a simple way to give short-term memory to our chat system.


## 🧠 Step 4 – Building a Chat with Memory (Part 1)

Let’s begin by setting up the **conversation history** list.  
This will hold all messages between you and the assistant.  
We’ll start with a system message that defines the assistant’s personality.


In [10]:
# %% Step 4.1 – Initialize the chat memory

# This list stores the conversation history
conversation_history = [
    {"role": "system", "content": "You are a helpful study assistant who explains clearly and politely."}
]

print("Conversation memory initialized.")


Conversation memory initialized.


In [11]:
# %% Step 4.2 – Get user input and check for exit command

user_input = input("👤 You: ")

if user_input.lower() in ["exit", "quit", "bye"]:
    print("👋 Chat ended.")
else:
    print("You said:", user_input)


You said: Hello world!


Next, we’ll **append** user message to the `conversation_history`  
and send the full conversation to the model.


In [12]:
# %% Step 4.3 – Add user's message and send to the model

conversation_history.append({"role": "user", "content": user_input})

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=conversation_history
)

assistant_message = response.choices[0].message.content
print("🤖 Assistant:", assistant_message)


🤖 Assistant: Hello! How can I assist you today?


Finally, we’ll store the assistant’s reply in the `conversation_history`.  
This helps the model remember the previous exchanges.


In [13]:
# %% Step 4.4 – Store assistant's reply for memory

conversation_history.append({"role": "assistant", "content": assistant_message})
print("Memory updated:", len(conversation_history), "messages in chat history.")


Memory updated: 3 messages in chat history.


### Combine Everything
If the above steps work, we can wrap them inside a `while True` loop later.  
That way, we can chat continuously until user type **exit**.

But for learning purposes, testing each cell one by one is better —  
you’ll understand how the conversation list evolves each time user send a message.


## Step 5 – Combine Everything into a Chat Loop

Now that we know each step works individually,  
we’ll combine them into a simple loop so you can chat continuously.  

Here’s what the loop will do:
1. Ask for user input.  
2. Add  message to the memory.  
3. Send the full chat history to the model.  
4. Print the model’s response.  
5. Save that response back into memory.  
6. Repeat until user type “exit”.


## Step 6 – Chat in the VS Code Terminal

Now we’ll make our chat loop a bit smoother:
- Add color for user vs. assistant text (optional, using emojis).
- Add short pauses so responses feel natural.
- Print a clean divider between turns.


## 🧠 Step 6.1 – Prepare the Conversation Memory
We'll start by creating the memory list that stores our chat.
The system message defines who the assistant is and sets the tone.


In [14]:
# %% Step 6.2 – Start chat loop
import time

print("Chat ready! Type 'exit' to stop.\n")

while True:
    user_input = input("👤 You: ")

    # Step 1: exit condition
    if user_input.lower() in ["exit", "quit", "bye"]:
        print("\n👋 Goodbye! Chat session ended.")
        break

    # Step 2: store user input in memory
    conversation_history.append({"role": "user", "content": user_input})


Chat ready! Type 'exit' to stop.


👋 Goodbye! Chat session ended.


In [15]:
import os
from dotenv import load_dotenv

# Use absolute Windows path
dotenv_path = r"C:\Users\rahul\Desktop\Project_DSI\deploying-ai\05_src\.secrets"
load_dotenv(dotenv_path)

print("Loaded key?", bool(os.getenv("OPENAI_API_KEY")))


Loaded key? True


In [16]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


In [17]:
from tools_api import get_stock_price    # Service 1
from tools_semantic import search_docs   # Service 2
from tools_custom import explain_simple  # Service 3

print("✅ All tools imported successfully!")


✅ All tools imported successfully!


In [18]:
print(search_docs("What is diversification?"))


Relevant info: Diversification reduces portfolio risk.


In [19]:
# --- Load Alpha Vantage API key ---
from dotenv import load_dotenv
import os

load_dotenv(r"C:\Users\rahul\Desktop\Project_DSI\deploying-ai\05_src\.secrets_f")
print("✅ Alpha Vantage key loaded:", bool(os.getenv("ALPHAVANTAGE_API_KEY")))


✅ Alpha Vantage key loaded: True


In [20]:
# --- Simple Command-Line Chat Loop ---

from openai import OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("💬 Finance Assistant Chat Started (type 'quit' to exit)")

while True:
    user_input = input("You: ")
    if user_input.lower() in ["quit", "exit"]:
        print("👋 Goodbye!")
        break

    # Step 2: route queries based on keywords
    if "price" in user_input or "stock" in user_input:
        response = get_stock_price(user_input)  # Service 1
    elif "meaning" in user_input or "define" in user_input:
        response = search_docs(user_input)       # Service 2
    else:
        response = explain_simple(user_input)    # Service 3

    print("Bot:", response)


💬 Finance Assistant Chat Started (type 'quit' to exit)


Bot: Today, the share price of Apple Inc. (AAPL) reflects how much investors think the company is worth. If the price goes up, people are optimistic about Apple's future. If it goes down, they may have concerns. The price can change every moment based on buyers and sellers. Always check a reliable financial news source for the latest number.
Bot: The share price of AAPL, which is Apple Inc.'s stock, is the current value of one share. It changes throughout the day based on buying and selling in the stock market. To find the exact price today, you can check financial news websites or stock market apps. Remember, share prices can rise or fall quickly!
Bot: AAPL is the stock symbol for Apple Inc. Its stock price shows how much one share of Apple is worth in the market. This price changes based on how many people want to buy or sell the stock. When lots of people want to buy Apple shares, the price goes up. When more people want to sell, the price goes down. The stock price reflects how inv

In [21]:
# --- Simple Command-Line Chat Loop ---

from openai import OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("💬 Finance Assistant Chat Started (type 'quit' to exit)")

while True:
    user_input = input("You: ")
    if user_input.lower() in ["quit", "exit"]:
        print("👋 Goodbye!")
        break

    # Step 2: route queries based on keywords
    if "price" in user_input or "stock" in user_input:
        response = get_stock_price(user_input)  # Service 1
    elif "meaning" in user_input or "define" in user_input:
        response = search_docs(user_input)       # Service 2
    else:
        response = explain_simple(user_input)    # Service 3

    print("Bot:", response)

💬 Finance Assistant Chat Started (type 'quit' to exit)
Bot: MSFT is currently trading at $496.82, with a daily change of -0.0563%.
Bot: The share price of GOOGL is the cost of one share of Alphabet Inc., the company that owns Google. It tells you how much you need to pay to buy a part of the company. Share prices can change every day based on how well the company performs and how investors feel about its future. You can find this price on financial news websites or stock market apps.
Bot: MSFT is currently trading at $496.82, with a daily change of -0.0563%.
Bot: The share price of a company, like T (AT&T), is the amount of money you pay to buy one share. It reflects what investors think the company is worth at a specific time. If the share price goes up, it means more people want to buy it. If it goes down, it means fewer people are interested. The share price changes every trading day based on supply and demand.
👋 Goodbye!


In [22]:
# --- Simple Command-Line Chat Loop ---

from openai import OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("💬 Finance Assistant Chat Started (type 'quit' to exit)")

while True:
    user_input = input("You: ")
    if user_input.lower() in ["quit", "exit"]:
        print("👋 Goodbye!")
        break

    # Step 2: route queries based on keywords
    if "price" in user_input or "stock" in user_input:
        response = get_stock_price(user_input)  # Service 1
    elif "meaning" in user_input or "define" in user_input:
        response = search_docs(user_input)       # Service 2
    else:
        response = explain_simple(user_input)    # Service 3

    print("Bot:", response)

💬 Finance Assistant Chat Started (type 'quit' to exit)
Bot: The stock price of AMZN, which is Amazon's stock, shows how much one share costs on the stock market. It changes every moment based on how many people want to buy or sell it. If more people want to buy, the price goes up. If more want to sell, the price goes down. You can find the current price by checking a financial news website or a stock app.
Bot: MSFT is currently trading at $496.82, with a daily change of -0.0563%.
Bot: I can't tell you the weather today because I don't have real-time data. To find out, you can check a weather app or website. They will give you information about temperature, rain, wind, and sunshine. If you need help understanding weather terms, feel free to ask!
Bot: A favorite animal is the one you like the most. Dogs are friendly and love to play. They need walks and attention. Cats are independent and enjoy cuddling. They are often playful and curious. Fish are calm and quiet, and they swim in water.

In [25]:
# --- Guardrails before routing ---

restricted_topics = ["cat", "cats", "dog", "dogs", "horoscope", "zodiac", "taylor swift","food"]

def contains_restricted_topic(text):
    """Check if user message contains any forbidden topics."""
    return any(topic in text.lower() for topic in restricted_topics)


In [26]:
while True:
    user_input = input("You: ")
    if user_input.lower() in ["quit", "exit"]:
        print("👋 Goodbye!")
        break

    # --- Guardrail check ---
    if contains_restricted_topic(user_input):
        print("Bot: Sorry, I can’t discuss that topic. Let’s talk about finance instead.")
        continue

    # --- Routing logic ---
    if "price" in user_input or "stock" in user_input:
        response = get_stock_price(user_input)
    elif "meaning" in user_input or "define" in user_input:
        response = search_docs(user_input)
    else:
        response = explain_simple(user_input)

    print("Bot:", response)


Bot: Sorry, I can’t discuss that topic. Let’s talk about finance instead.
👋 Goodbye!


In [36]:
def chat_with_bot(message, history):
    # --- Guardrails ---
    restricted_topics = ["cat", "cats", "dog", "dogs", "horoscope", "zodiac", "taylor swift"]
    if any(word in message.lower() for word in restricted_topics):
        return "🚫 Sorry, I can’t talk about that topic."

    # --- Memory handling ---
    context = "\n".join([f"User: {u}\nBot: {b}" for u, b in history])  # summarize past messages

    # --- Routing logic ---
    if "price" in message or "stock" in message:
        response = get_stock_price(message)
    elif "meaning" in message or "define" in message:
        response = search_docs(message)
    else:
        response = explain_simple(message)

    # --- Optional: add contextual tag ---
    response = f"(Based on previous chat)\n{response}" if len(history) > 0 else response
    return response


In [49]:
def chat_with_bot(message, history):
    # --- Guardrails: prevent restricted topics ---
    restricted_topics = ["cat", "cats", "dog", "dogs", "horoscope", "zodiac", "taylor swift"]
    if any(word in message.lower() for word in restricted_topics):
        return "🚫 Sorry, I can’t talk about that topic."

    # --- Build short-term memory context ---
    recent_history = history[-3:]  # remember last 3 turns
    context = "\n".join([f"User: {u}\nBot: {b}" for u, b in recent_history])

    # --- Decide which service to call ---
    if "price" in message.lower() or "stock" in message.lower():
        response = get_stock_price(message)              # Service 1: API (Alpha Vantage)
    elif "meaning" in message.lower() or "define" in message.lower():
        response = search_docs(message)                  # Service 2: Semantic search
    elif "mcp" in message.lower() or "yahoo" in message.lower():
        import asyncio
        response = asyncio.run(yahoo_finance_search(message))  # Service 3: Simulated MCP
    else:
        response = explain_simple(message)               # Fallback / custom service

    # --- Add memory context tag for readability ---
    if len(history) > 0:
        response = f"(Based on previous chat)\n{response}"

    return response



In [50]:


# --- Gradio Chat Interface ---
import gradio as gr

demo = gr.ChatInterface(
    fn=chat_with_bot,
    title="Finance Assistant",
    description="Ask me about stocks, definitions, or simple finance terms.",
)

demo.launch(share=True)



c:\Users\rahul\Desktop\Project_DSI\deploying-ai\deploying-ai-env\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7872
* Running on public URL: https://10d05b8e9a8ef765c1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [52]:
# --- Gradio Chat Interface ---
import gradio as gr

# Create a chat interface using your existing function
chat_ui = gr.ChatInterface(
    fn=chat_with_bot,          # the brain (your main chat function)
    title="Your Personal Finance Support",
    description="Ask about stock prices, finance terms, or try the MCP Yahoo Finance tool!",
    examples=[
        ["What is diversification?"],
        ["Get stock price of MSFT"],
        ["Use MCP to get Tesla data"],
        ["Explain liquidity in simple terms"],
    ],
    theme="soft",  # gives a clean, beginner-friendly look
)

# Launch the chat
chat_ui.launch(share=True)


c:\Users\rahul\Desktop\Project_DSI\deploying-ai\deploying-ai-env\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7874
* Running on public URL: https://20c04bb7e4388dda29.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Enhanced Request

In [62]:
from memory import add_to_memory, get_recent_memory

def chat_with_bot(message, history):
    # Restricted topics guard
    restricted_topics = ["cat", "cats", "dog", "dogs", "horoscope", "zodiac", "taylor swift"]
    if any(word in message.lower() for word in restricted_topics):
        return "🚫 Sorry, I can’t talk about that topic."

    # Retrieve context
    context = get_recent_memory()

    # Route query
    if "price" in message or "stock" in message:
        response = get_stock_price(message)
    elif "meaning" in message or "define" in message:
        response = search_docs(message)
    else:
        response = explain_simple(message)

    # Update memory
    add_to_memory(message, response)

    # Return response
    return f"(Context memory active)\n{response}"


In [71]:
# --- Gradio Chat Interface with Animated Guardio Emoji ---
import gradio as gr

# --- Expanded Restricted Topics ---
guardrail_categories = {
    "animals": ["cat", "cats", "dog", "dogs", "puppy", "kitten"],
    "astrology": ["horoscope", "zodiac", "aries", "leo", "scorpio"],
    "celebrities": ["taylor swift", "beyonce", "kanye west"],
    "sensitive": ["politics", "religion", "violence", "drugs", "adult", "sex"]
}

# --- Stateful emoji toggle ---
current_emoji = "💹"

# --- Enhanced Gradio Interface with Clear Memory Button ---
def guardrails_wrapper(message, history):
    """Wraps the chat function with guardrails and memory context."""
    restricted_topics = ["cat", "cats", "dog", "dogs", "horoscope", "zodiac", "taylor swift"]
    if any(word in message.lower() for word in restricted_topics):
        return "🛡️ Sorry, I can’t discuss that topic. Let’s stay on finance. 💹"
    
    # Use memory to add context
    context = get_recent_memory()

    if "price" in message or "stock" in message:
        response = get_stock_price(message)
    elif "meaning" in message or "define" in message:
        response = search_docs(message)
    else:
        response = explain_simple(message)

    add_to_memory(message, response)
    return f"(Context active)\n{response}"


# --- Clear memory logic ---
def on_clear_memory():
    """Called when the user clicks the 🧹 button."""
    clear_memory()
    return "🧠 Memory cleared! The assistant is starting fresh."


# --- Build the Gradio UI ---
with gr.Blocks(title="💹 Finance Support (Guardio & Memory)") as demo:
    gr.Markdown(
        "### 💬 Finance Support (Guardio 🛡️ Enabled)\n"
        "Ask about stocks, finance terms, or MCP simulations.\n\n"
        "🛡️ **Guardio Notice:** This assistant automatically filters restricted or sensitive topics "
        "to maintain a safe, educational chat environment.\n\n"
        "🧠 Memory stores your last 5 exchanges.\n"
        "🧹 Use the **Clear Memory** button below to reset conversation history."
    )

    chat = gr.ChatInterface(fn=guardrails_wrapper)

    clear_btn = gr.Button("🧹 Clear Memory")
    clear_output = gr.Textbox(label="System Message")

    clear_btn.click(fn=on_clear_memory, outputs=clear_output)

demo.launch(share=True)




c:\Users\rahul\Desktop\Project_DSI\deploying-ai\deploying-ai-env\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7891
* Running on public URL: https://0ba494191795ed95ec.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
